# 面试题：为什么 Agent 必须权威状态回读？

面试回答：模型工作记忆和缓存是推理层，权威系统读取才是事实层。高风险提交、超时重试与宣称完成前要检查资源 ID、版本、更新时间和读权限。发现版本变化应重新计算而非覆盖最新事实；回读也可能最终一致，因此要有 pending、轮询窗口和人工升级。

## 真实案例

支付 Agent 从缓存读取订单可退款余额后，人工后台已经处理一部分退款。六个事件展示旧记忆、权威回读、版本一致和陈旧快照。

## 基线

基线直接按缓存余额提交退款。

## 结果解读

手写 guard 输出缓存版本、权威版本、可退款余额和决策。

## 失败案例

缓存显示 100，权威余额已变为 20；继续退款 80 会超额。

In [1]:
requests = [{'id':'Q1','cached_balance':100,'cached_version':5,'authoritative_balance':100,'authoritative_version':5,'amount':80}, {'id':'Q2','cached_balance':100,'cached_version':5,'authoritative_balance':20,'authoritative_version':6,'amount':80}, {'id':'Q3','cached_balance':50,'cached_version':3,'authoritative_balance':50,'authoritative_version':3,'amount':30}, {'id':'Q4','cached_balance':40,'cached_version':7,'authoritative_balance':0,'authoritative_version':8,'amount':10}, {'id':'Q5','cached_balance':60,'cached_version':4,'authoritative_balance':60,'authoritative_version':4,'amount':60}, {'id':'Q6','cached_balance':10,'cached_version':2,'authoritative_balance':10,'authoritative_version':2,'amount':15}]  # 构造六条退款请求，包含缓存与权威快照。
print('退款读写输入:', requests)  # 输出缓存、版本、权威余额与请求金额。
print('教学说明：authoritative 字段代表支付服务强一致读取，cache 只是 Agent 先前记忆。')  # 明确事实层和推理层。

退款读写输入: [{'id': 'Q1', 'cached_balance': 100, 'cached_version': 5, 'authoritative_balance': 100, 'authoritative_version': 5, 'amount': 80}, {'id': 'Q2', 'cached_balance': 100, 'cached_version': 5, 'authoritative_balance': 20, 'authoritative_version': 6, 'amount': 80}, {'id': 'Q3', 'cached_balance': 50, 'cached_version': 3, 'authoritative_balance': 50, 'authoritative_version': 3, 'amount': 30}, {'id': 'Q4', 'cached_balance': 40, 'cached_version': 7, 'authoritative_balance': 0, 'authoritative_version': 8, 'amount': 10}, {'id': 'Q5', 'cached_balance': 60, 'cached_version': 4, 'authoritative_balance': 60, 'authoritative_version': 4, 'amount': 60}, {'id': 'Q6', 'cached_balance': 10, 'cached_version': 2, 'authoritative_balance': 10, 'authoritative_version': 2, 'amount': 15}]
教学说明：authoritative 字段代表支付服务强一致读取，cache 只是 Agent 先前记忆。


In [2]:
def cache_only(row):  # 定义不做权威回读的危险基线。
    return '提交退款' if row['cached_balance'] >= row['amount'] else '拒绝退款'  # 仅依据工作记忆判断可执行性。
baseline = [(row['id'], cache_only(row)) for row in requests]  # 对六条请求运行缓存决策。
print('缓存基线:', baseline)  # 输出 Q2 会依据过期余额错误提交。

缓存基线: [('Q1', '提交退款'), ('Q2', '提交退款'), ('Q3', '提交退款'), ('Q4', '提交退款'), ('Q5', '提交退款'), ('Q6', '拒绝退款')]


In [3]:
def readback_guard(row):  # 定义提交前的权威状态回读门禁。
    stale = row['cached_version'] != row['authoritative_version']  # 判断 Agent 记忆版本是否已过期。
    enough = row['authoritative_balance'] >= row['amount']  # 依据权威余额检查退款上限。
    if stale:  # 对版本变化的资源不沿用旧计划。
        return 'replan_after_readback', {'stale':stale,'balance':row['authoritative_balance'],'enough':enough}  # 要求用新事实重算动作。
    if not enough:  # 处理版本一致但余额不足的明确业务拒绝。
        return 'reject_insufficient', {'stale':stale,'balance':row['authoritative_balance'],'enough':enough}  # 返回可解释的领域错误。
    return 'submit_with_version', {'stale':stale,'balance':row['authoritative_balance'],'enough':enough}  # 将当前版本作为写入前置条件提交。

In [4]:
results = [(row['id'],) + readback_guard(row) for row in requests]  # 对六条请求执行提交前权威回读。
print('id | 回读决策 | 证据')  # 输出权威门禁结果表标题。
for item in results:  # 遍历每条请求的版本、余额和最终决策。
    print(item[0], item[1], item[2])  # 输出可审计的中间量。
print('陈旧缓存数:', sum(item[2]['stale'] for item in results))  # 汇总需要重新规划的缓存过期请求。

id | 回读决策 | 证据
Q1 submit_with_version {'stale': False, 'balance': 100, 'enough': True}
Q2 replan_after_readback {'stale': True, 'balance': 20, 'enough': False}
Q3 submit_with_version {'stale': False, 'balance': 50, 'enough': True}
Q4 replan_after_readback {'stale': True, 'balance': 0, 'enough': False}
Q5 submit_with_version {'stale': False, 'balance': 60, 'enough': True}
Q6 reject_insufficient {'stale': False, 'balance': 10, 'enough': False}
陈旧缓存数: 2


In [5]:
wrong = dict(baseline)['Q2']  # 读取 Q2 按旧缓存得出的错误提交。
fixed = dict((item[0], item[1]) for item in results)['Q2']  # 读取 Q2 权威回读后的重规划结论。
print('失败案例 Q2：缓存=', wrong, '，权威回读=', fixed)  # 展示旧记忆不能作为高风险写依据。
print('生产差距：需配置资源 TTL、一致性级别、ETag 条件写、回读超时策略、缓存来源时间和读权限审计。')  # 说明权威读设计要素。

失败案例 Q2：缓存= 提交退款 ，权威回读= replan_after_readback
生产差距：需配置资源 TTL、一致性级别、ETag 条件写、回读超时策略、缓存来源时间和读权限审计。


In [6]:
assert readback_guard(requests[0])[0] == 'submit_with_version'  # 验证同版本且余额充足可提交条件写。
assert readback_guard(requests[1])[0] == 'replan_after_readback'  # 验证陈旧缓存会触发重规划而非写入。
assert readback_guard(requests[5])[0] == 'reject_insufficient'  # 验证余额不足即使缓存新鲜也会被拒绝。